# Arabic Semantic Space Lab — GATE

**Objective:**
Create eight semantic examples within a single domain, then test whether
`Omartificial-Intelligence-Space/GATE-AraBert-v1` places the positive sentence closer to the anchor sentence than the negative sentences.

كل مثال يحتوي على:

- **Anchor:** الجملة الأصلية.
- **Positive:** نفس المعنى بصياغة مختلفة.
- **Hard negative:** كلمات/موضوع قريب، لكن المعنى مختلف في نقطة مهمة.
- **Easy negative:** معنى واضح الاختلاف.

في النهاية ستحصل على خريطة تفاعلية، مصفوفة تشابه، أقرب الجمل، وتحميل بياناتك بصيغتي JSONL وCSV.

> لا تستخدم أسماء حقيقية، أرقام هويات/حسابات، بيانات صحية حقيقية، أسرار عمل، أو نصوصاً منسوخة.

## قواعد جودة البيانات

1. اكتب الجمل بنفسك وبالعربية الطبيعية.
2. حافظ الـPositive على **كل** المعنى، وليس الموضوع فقط.
3. اجعل الـHard negative قريباً لفظياً أو موضوعياً، مع تغيير دلالي مهم: نفي، رقم، زمن، كيان، شرط، سماح/منع، أو حالة مكتملة/معلّقة.
4. اجعل الـEasy negative مختلفاً بوضوح.
5. لا تكرر الجمل بين الأمثلة.
6. استخدم آلية Hard negative متنوعة عبر الأمثلة الثمانية.

In [ ]:
%pip -q install -U sentence-transformers umap-learn plotly pandas scikit-learn

In [ ]:
import json, re, warnings
from pathlib import Path

import numpy as np
import pandas as pd
import plotly.express as px
import plotly.graph_objects as go
from sklearn.metrics.pairwise import cosine_similarity

warnings.filterwarnings("ignore")
pd.set_option("display.max_colwidth", 120)

MODEL_ID = "Omartificial-Intelligence-Space/GATE-AraBert-v1"
MODEL_REVISION = "e8537da79d1870992cd573094ef194cf2d151a73"
N_TRIPLETS = 8
ROLES = ["anchor", "positive", "hard_negative", "easy_negative"]
ROLE_AR = {
    "anchor": "الأصلية", "positive": "الموجبة",
    "hard_negative": "السالبة الصعبة", "easy_negative": "السالبة السهلة"
}
print("جاهز ✅")

## 1) بيانات المشارك والمجال

استخدم رمز المشارك الذي أعطاك إياه المدرب، واختر **مجالاً واحداً** لكل الأمثلة.

In [ ]:
PARTICIPANT_ID = "P066"       # مثال: P012
DOMAIN = "Veterinary Care"          # المجال المخصص لك بالإنجليزية
DOMAIN_AR = "الرعاية البيطرية"   # اسم المجال بالعربية

assert re.fullmatch(r"P\d{3}", PARTICIPANT_ID), "استخدم رمزاً مثل P012"
assert DOMAIN.strip() and DOMAIN_AR.strip(), "أدخل المجال بالعربية والإنجليزية"
print(f"المشارك: {PARTICIPANT_ID} | المجال: {DOMAIN_AR}")

## 2) مثال توضيحي — لا يُضاف إلى بياناتك

لاحظ أن السالب الصعب يستخدم كلمات قريبة لكنه يقلب المعنى بواسطة **تجنب**.

In [ ]:
DEMO = {
    "anchor": "يجب على المريض تناول الدواء بعد الطعام",
    "positive": "ينبغي للمريض أخذ العلاج عقب تناول الوجبة",
    "hard_negative": "يجب على المريض تجنب تناول الدواء بعد الطعام",
    "easy_negative": "تأخرت رحلة الطيران بسبب الأحوال الجوية",
    "hard_negative_mechanism": "negation_or_reversal",
    "difficulty": "hard",
    "naturalness": 5,
    "notes": "تجنّب تقلب الإجراء المطلوب"
}
pd.DataFrame([DEMO]).T.rename(columns={0: "المثال"})

## 3) اكتب أمثلتك الثمانية

الآليات المسموحة للسالب الصعب:
`negation`, `number`, `time`, `entity`, `condition`, `reversed_relation`, `permission_prohibition`, `status_intent`, `other`.

استبدل النصوص الفارغة. يجب أن تكون `naturalness` من 1 إلى 5.

In [ ]:
# ============================================================
# PARTICIPANT DATA ENTRY
# Create 8 semantic triplets within your assigned domain.
# Replace every "..." with your own Arabic sentence or answer.
# ============================================================

TRIPLETS = [
    # Triplet 1
    {
        "anchor": "فحص الطبيب البيطري الكلب قبل إعطائه العلاج",
        "positive": "عاين الطبيب البيطري الكلب قبل وصف العلاج له",
        "hard_negative": "لم يفحص الطبيب البيطري الكلب قبل إعطائه العلاج",
        "easy_negative": "ازدحم الطريق بسبب أعمال الصيانة",
        "hard_negative_mechanism": "negation",
        "difficulty": "hard",
        "naturalness": 5,
        "notes": "..."
    },

    # Triplet 2
    {
        "anchor": "أعطى الطبيب البيطري القطة جرعتين من اللقاح",
        "positive": "قام الطبيب البيطري بإعطاء القطة جرعتين من التطعيم",
        "hard_negative": "أعطى الطبيب البيطري القطة ثلاث جرعات من اللقاح",
        "easy_negative": "اشترى أحمد هاتفًا جديدًا",
        "hard_negative_mechanism": "changed_number",
        "difficulty": "hard",
        "naturalness": 5,
        "notes": "..."
    },

    # Triplet 3
    {
        "anchor": "بدأ الطبيب البيطري العملية بعد تخدير القطة بالكامل",
        "positive": "أجرى الطبيب البيطري الجراحة بعد تخدير القطة تخديرًا كاملًا",
        "hard_negative": "بدأ الطبيب البيطري العملية قبل تخدير القطة بالكامل",
        "easy_negative": "سافر خالد إلى دبي في عطلة الصيف",
        "hard_negative_mechanism": "changed_time",
        "difficulty": "hard",
        "naturalness": 5,
        "notes": "..."
    },

    # Triplet 4
    {
        "anchor": "أكد الطبيب البيطري أن نتائج الفحوصات تسمح بإجراء الجراحة",
        "positive": "أوضح الطبيب البيطري أن نتائج التحاليل تسمح بإجراء العملية",
        "hard_negative": "أكد الطبيب البيطري أن نتائج الفحوصات لا تسمح بإجراء الجراحة",
        "easy_negative": "حقق المنتخب الوطني الفوز في المباراة",
        "hard_negative_mechanism": "negation",
        "difficulty": "hard",
        "naturalness": 5,
        "notes": "..."
    },

    # Triplet 5
    {
        "anchor": "وصف الطبيب البيطري علاج الحساسية للأرنب",
        "positive": "أوصى الطبيب البيطري بدواء لعلاج حساسية الأرنب",
        "hard_negative": "وصف الطبيب البيطري علاج العدوى للأرنب",
        "easy_negative": "ازداد عدد السياح هذا العام.",
        "hard_negative_mechanism": "changed_entity",
        "difficulty": "hard",
        "naturalness": 5,
        "notes": "..."
    },

    # Triplet 6
    {
        "anchor": "سلّم الطبيب البيطري نتائج الفحوصات لصاحب الحصان",
        "positive": "أعطى الطبيب البيطري تقرير الفحوصات لمالك الحصان",
        "hard_negative": "سلّم صاحب الحصان نتائج الفحوصات للطبيب البيطري",
        "easy_negative": "بدأت الأمطار تهطل منذ الصباح",
        "hard_negative_mechanism": "reversed_relationship",
        "difficulty": "medium",
        "naturalness": 5,
        "notes": "..."
    },

    # Triplet 7
    {
        "anchor": "أكد الطبيب البيطري أن الكلب لا يحتاج إلى عملية جراحية",
        "positive": "أوضح الطبيب البيطري أن الكلب ليس بحاجة إلى تدخل جراحي",
        "hard_negative": "أكد الطبيب البيطري أن الكلب يحتاج إلى عملية جراحية",
        "easy_negative": "أبحرت السفينة نحو الميناء صباحًا",
        "hard_negative_mechanism": "negation",
        "difficulty": "medium",
        "naturalness": 5,
        "notes": "..."
    },

    # Triplet 8
    {
        "anchor": "يجب إعطاء القطة نصف قرص يوميًا",
        "positive": "ينبغي أن تتناول القطة نصف حبة كل يوم",
        "hard_negative": "يجب إعطاء القطة قرصًا كاملًا يوميًا",
        "easy_negative": "أعلن النادي عن مدربه الجدي",
        "hard_negative_mechanism": "changed_number",
        "difficulty": "medium",
        "naturalness": 5,
        "notes": "..."
    }
]

# Allowed values for hard_negative_mechanism:
# negation, changed_number, changed_time, changed_location,
# changed_entity, changed_condition, reversed_relationship,
# changed_intent, permission_prohibition, completed_pending, other

# Allowed values for difficulty:
# easy, medium, hard

# naturalness must be an integer from 1 to 5.

# Display the entered data as a table
import pandas as pd

triplets_df = pd.DataFrame(TRIPLETS)
triplets_df

## 4) التحقق من الجودة

لن يبدأ التحليل حتى تكتمل الأمثلة الثمانية وتنجح قواعد التحقق.

In [ ]:
ALLOWED_MECHANISMS = {
    "negation",
    "changed_number",
    "changed_time",
    "changed_location",
    "changed_entity",
    "changed_condition",
    "reversed_relationship",
    "changed_intent",
    "permission_prohibition",
    "completed_pending",
    "other"
}
ALLOWED_DIFFICULTY = {"easy", "medium", "hard"}
ARABIC_RE = re.compile(r"[\u0600-\u06FF]")

def validate_triplets(rows):
    errors, seen = [], {}
    if len(rows) != N_TRIPLETS:
        errors.append(f"يجب إدخال {N_TRIPLETS} أمثلة بالضبط")
    for i, row in enumerate(rows, 1):
        for field in ROLES:
            value = row.get(field)
            if not isinstance(value, str) or not value.strip():
                errors.append(f"المثال {i}: الحقل {field} فارغ")
                continue
            text = " ".join(value.split())
            if len(text) < 12:
                errors.append(f"المثال {i}: {field} قصير جداً")
            if not ARABIC_RE.search(text):
                errors.append(f"المثال {i}: {field} لا يحتوي نصاً عربياً")
            key = re.sub(r"\s+", " ", text).strip()
            if key in seen:
                errors.append(f"الجملة مكررة في المثالين {seen[key]} و{i}")
            else:
                seen[key] = i
        if row.get("anchor", "").strip() == row.get("positive", "").strip():
            errors.append(f"المثال {i}: الـPositive نسخة مطابقة للـAnchor")
        if row.get("hard_negative_mechanism") not in ALLOWED_MECHANISMS:
            errors.append(f"المثال {i}: آلية السالب الصعب غير صحيحة")
        if row.get("difficulty") not in ALLOWED_DIFFICULTY:
            errors.append(f"المثال {i}: difficulty يجب أن تكون easy/medium/hard")
        if row.get("naturalness") not in range(1, 6):
            errors.append(f"المثال {i}: naturalness يجب أن تكون من 1 إلى 5")
    if errors:
        raise ValueError("فشل التحقق:\n- " + "\n- ".join(errors[:40]))
    return True

validate_triplets(TRIPLETS)
print("نجح التحقق من البيانات ✅")

## 5) تحميل GATE وإنشاء التضمينات

ينتج النموذج متجهاً من 768 بُعداً لكل جملة. نطبّع المتجهات، ثم يصبح حاصل الضرب بينها هو **Cosine similarity**.

In [ ]:
from sentence_transformers import SentenceTransformer

model = SentenceTransformer(MODEL_ID, revision=MODEL_REVISION)

records = []
for i, row in enumerate(TRIPLETS, 1):
    triplet_id = f"{PARTICIPANT_ID}_T{i:02d}"
    for role in ROLES:
        records.append({
            "participant_id": PARTICIPANT_ID,
            "triplet_id": triplet_id,
            "domain": DOMAIN,
            "domain_ar": DOMAIN_AR,
            "role": role,
            "text": " ".join(row[role].split()),
            "hard_negative_mechanism": row["hard_negative_mechanism"],
            "difficulty": row["difficulty"],
            "naturalness": row["naturalness"],
            "notes": row.get("notes", "")
        })

sentences = [r["text"] for r in records]
embeddings = model.encode(
    sentences, batch_size=32, show_progress_bar=True,
    convert_to_numpy=True, normalize_embeddings=True
)
print(f"تمثيل {len(sentences)} جملة ← {embeddings.shape[1]} بُعداً ✅")

## 6) هل نجح كل مثال؟

نستخدم معيارين:

- **النجاح الأساسي:** $sim(A,P) > sim(A,HN)$ و $sim(A,P) > sim(A,EN)$.
- **الترتيب الكامل (أصعب):** $sim(A,P) > sim(A,HN) > sim(A,EN)$.

السالب الصعب قد يكون أحياناً أبعد من السالب السهل؛ لذلك لا نستخدم الترتيب الكامل وحده للحكم.

In [ ]:
rows = []
for i, row in enumerate(TRIPLETS):
    start = i * 4
    A, P, HN, EN = embeddings[start:start+4]
    ap, ah, ae = float(A @ P), float(A @ HN), float(A @ EN)
    rows.append({
        "triplet_id": f"{PARTICIPANT_ID}_T{i+1:02d}",
        "sim_anchor_positive": ap,
        "sim_anchor_hard_negative": ah,
        "sim_anchor_easy_negative": ae,
        "positive_margin_over_hard": ap - ah,
        "positive_margin_over_easy": ap - ae,
        "triplet_success": ap > ah and ap > ae,
        "full_order_success": ap > ah > ae,
        "hard_negative_mechanism": row["hard_negative_mechanism"],
        "difficulty": row["difficulty"]
    })

score_df = pd.DataFrame(rows)
display(score_df.style.format({c: "{:.3f}" for c in score_df.columns if c.startswith(("sim_", "positive_"))}))
print(f"النجاح الأساسي: {score_df.triplet_success.mean():.1%}")
print(f"الترتيب الكامل: {score_df.full_order_success.mean():.1%}")

## 7) مقارنة درجات التشابه

In [ ]:
long_scores = score_df.melt(
    id_vars="triplet_id",
    value_vars=["sim_anchor_positive", "sim_anchor_hard_negative", "sim_anchor_easy_negative"],
    var_name="relation", value_name="cosine_similarity"
)
label_map = {
    "sim_anchor_positive": "Anchor–Positive",
    "sim_anchor_hard_negative": "Anchor–Hard negative",
    "sim_anchor_easy_negative": "Anchor–Easy negative"
}
long_scores["relation"] = long_scores["relation"].map(label_map)
fig = px.bar(long_scores, x="triplet_id", y="cosine_similarity", color="relation", barmode="group",
             range_y=[-0.1, 1], title="Cosine similarity لكل مثال")
fig.update_layout(xaxis_title="المثال", yaxis_title="Cosine similarity", legend_title="العلاقة")
fig.show()

## 8) خريطة الفضاء الدلالي ثنائية الأبعاد

UMAP/PCA أدوات عرض فقط: القرب في الخريطة قد يتشوّه عند ضغط 768 بُعداً إلى بُعدين. اعتمد درجات cosine الأصلية للحكم الدقيق.

In [ ]:
try:
    import umap
    reducer = umap.UMAP(
        n_components=2, n_neighbors=min(10, len(sentences)-1),
        min_dist=0.15, metric="cosine", random_state=42
    )
    xy = reducer.fit_transform(embeddings)
    projection_name = "UMAP"
except Exception as exc:
    from sklearn.decomposition import PCA
    xy = PCA(n_components=2, random_state=42).fit_transform(embeddings)
    projection_name = "PCA"
    print("تعذر UMAP؛ تم استخدام PCA:", exc)

viz_df = pd.DataFrame(records)
viz_df["x"], viz_df["y"] = xy[:, 0], xy[:, 1]
viz_df["role_ar"] = viz_df.role.map(ROLE_AR)

fig = px.scatter(
    viz_df, x="x", y="y", color="triplet_id", symbol="role",
    hover_data={"text": True, "role_ar": True, "x": ":.3f", "y": ":.3f"},
    title=f"{projection_name}: الفضاء الدلالي لجمل {DOMAIN_AR}"
)
for triplet_id, group in viz_df.groupby("triplet_id"):
    anchor = group[group.role == "anchor"].iloc[0]
    positive = group[group.role == "positive"].iloc[0]
    fig.add_trace(go.Scatter(
        x=[anchor.x, positive.x], y=[anchor.y, positive.y], mode="lines",
        line=dict(color="rgba(80,80,80,.35)", width=1),
        hoverinfo="skip", showlegend=False
    ))
fig.update_traces(marker=dict(size=12, line=dict(width=1, color="white")))
fig.update_layout(xaxis_title=f"{projection_name}-1", yaxis_title=f"{projection_name}-2")
fig.show()

## 9) مصفوفة التشابه لمثال واحد

غيّر `SELECTED_TRIPLET` من 1 إلى 8.

In [ ]:
SELECTED_TRIPLET = 1
assert 1 <= SELECTED_TRIPLET <= N_TRIPLETS
start = (SELECTED_TRIPLET - 1) * 4
local_embeddings = embeddings[start:start+4]
matrix = cosine_similarity(local_embeddings)
labels = [ROLE_AR[r] for r in ROLES]
fig = px.imshow(matrix, x=labels, y=labels, text_auto=".3f", zmin=-1, zmax=1,
                color_continuous_scale="RdBu_r", title=f"مصفوفة التشابه — المثال {SELECTED_TRIPLET}")
fig.show()

## 10) أقرب الجمل

اختر رقم أي جملة من الجدول، ثم راقب هل يعيد النموذج الجملة الموجبة الصحيحة أم جملة أخرى.

In [ ]:
sentence_table = viz_df[["triplet_id", "role", "text"]].copy()
sentence_table.index.name = "sentence_index"
display(sentence_table)

QUERY_INDEX = 0
TOP_K = 5
all_sim = embeddings @ embeddings[QUERY_INDEX]
neighbors = np.argsort(-all_sim)
neighbors = [i for i in neighbors if i != QUERY_INDEX][:TOP_K]
neighbor_df = sentence_table.iloc[neighbors].copy()
neighbor_df.insert(0, "cosine_similarity", all_sim[neighbors])
print("الاستعلام:", sentence_table.iloc[QUERY_INDEX].text)
display(neighbor_df)